In [3]:
# =========================
# 🚀 ONE-CLICK SHM PIPELINE (UPDATED)
# =========================

import os
os.system("pip install pandas scikit-learn joblib numpy")

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

print("🔄 Starting pipeline...\n")

# =========================
# 1. LOAD DATA
# =========================
data = pd.read_csv("cantilever_features.csv")

print("📊 Data Loaded. Shape:", data.shape)

# =========================
# 2. PREPROCESSING
# =========================

# Drop rows with missing values
data = data.dropna()

# Rename label column
data = data.rename(columns={'damage_label': 'Label'})

# ❌ Drop ONLY irrelevant columns
data = data.drop(['sample_id'], axis=1)

# ⚠️ Encode 'session' (important if different experiments)
data['session'] = LabelEncoder().fit_transform(data['session'])

# Encode target
le = LabelEncoder()
data['Label_encoded'] = le.fit_transform(data['Label'])

# Split features & target
X = data.drop(['Label', 'Label_encoded'], axis=1)
y = data['Label_encoded']

features = X.columns.tolist()

print("✅ Features used:")
print(features, "\n")

# =========================
# 3. TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# =========================
# 4. SCALING
# =========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================
# 5. MODEL TRAINING
# =========================
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42
)

model.fit(X_train_scaled, y_train)

# =========================
# 6. EVALUATION
# =========================
pred = model.predict(X_test_scaled)

print("📊 Accuracy:", accuracy_score(y_test, pred))
print("📊 Confusion Matrix:\n", confusion_matrix(y_test, pred), "\n")

# =========================
# 7. FEATURE IMPORTANCE (🔥 IMPORTANT FOR VIVA)
# =========================
importance = model.feature_importances_

feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': importance
}).sort_values(by='Importance', ascending=False)

print("🔥 Top Important Features:\n")
print(feature_importance.head(10), "\n")

# =========================
# 8. SAVE EVERYTHING
# =========================
joblib.dump(model, "shm_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(le, "label_encoder.pkl")
joblib.dump(features, "features.pkl")

print("💾 Model & components saved\n")

# =========================
# 9. SAMPLE PREDICTION
# =========================
sample_input = X.iloc[0:1]  # real sample from dataset

sample_scaled = scaler.transform(sample_input)
prediction = model.predict(sample_scaled)
label = le.inverse_transform(prediction)

print("🔍 Sample Prediction:", label[0])

# =========================
print("\n✅ PIPELINE COMPLETED SUCCESSFULLY")

🔄 Starting pipeline...

📊 Data Loaded. Shape: (1438, 23)
✅ Features used:
['session', 'damage_level', 'RMS', 'Std', 'Kurtosis', 'Skewness', 'Peak', 'Crest_Factor', 'Shape_Factor', 'Energy', 'Natural_Freq', 'Spec_Centroid', 'Band_Low', 'Band_Natural', 'Band_High', 'Spec_Entropy', 'AR1', 'AR2', 'AR_Resid_Std', 'Corr_XZ', 'Corr_YZ'] 

📊 Accuracy: 1.0
📊 Confusion Matrix:
 [[72  0  0  0]
 [ 0 72  0  0]
 [ 0  0 72  0]
 [ 0  0  0 72]] 

🔥 Top Important Features:

         Feature  Importance
1   damage_level    0.444582
0        session    0.429363
4       Kurtosis    0.022216
8   Shape_Factor    0.011351
15  Spec_Entropy    0.009035
7   Crest_Factor    0.007627
19       Corr_XZ    0.007248
6           Peak    0.006606
5       Skewness    0.006537
20       Corr_YZ    0.005818 

💾 Model & components saved

🔍 Sample Prediction: Healthy

✅ PIPELINE COMPLETED SUCCESSFULLY


In [4]:
# =========================
# 🔍 MULTIPLE TEST INPUTS (FIXED)
# =========================

# Create realistic test inputs (same structure as training data)
test_inputs = pd.DataFrame([
    {
        'session': 0,
        'damage_level': 0,
        'RMS': 36.7, 'Std': 36.7, 'Kurtosis': 0.28, 'Skewness': 0.08,
        'Peak': 117.1, 'Crest_Factor': 3.18, 'Shape_Factor': 1.27,
        'Energy': 270000, 'Natural_Freq': 5.5, 'Spec_Centroid': 25.0,
        'Band_Low': 0.17, 'Band_Natural': 0.29, 'Band_High': 0.53,
        'Spec_Entropy': 4.18, 'AR1': -0.01, 'AR2': -0.04,
        'AR_Resid_Std': 36.7, 'Corr_XZ': -0.03, 'Corr_YZ': -0.06
    },

    {
        'session': 0,
        'damage_level': 1,
        'RMS': 30.0, 'Std': 30.0, 'Kurtosis': 1.2, 'Skewness': 0.5,
        'Peak': 95.0, 'Crest_Factor': 3.8, 'Shape_Factor': 1.5,
        'Energy': 200000, 'Natural_Freq': 12.0, 'Spec_Centroid': 30.0,
        'Band_Low': 0.2, 'Band_Natural': 0.4, 'Band_High': 0.4,
        'Spec_Entropy': 4.5, 'AR1': 0.1, 'AR2': -0.1,
        'AR_Resid_Std': 30.5, 'Corr_XZ': 0.2, 'Corr_YZ': 0.1
    },

    {
        'session': 0,
        'damage_level': 2,
        'RMS': 20.0, 'Std': 20.0, 'Kurtosis': 3.5, 'Skewness': 1.5,
        'Peak': 60.0, 'Crest_Factor': 5.5, 'Shape_Factor': 2.5,
        'Energy': 120000, 'Natural_Freq': 40.0, 'Spec_Centroid': 45.0,
        'Band_Low': 0.3, 'Band_Natural': 0.3, 'Band_High': 0.4,
        'Spec_Entropy': 5.0, 'AR1': 0.3, 'AR2': -0.3,
        'AR_Resid_Std': 22.0, 'Corr_XZ': 0.5, 'Corr_YZ': 0.4
    }
])

# Ensure correct column order
test_inputs = test_inputs[features]

# =========================
# SCALE
# =========================
scaled_inputs = scaler.transform(test_inputs)

# =========================
# PREDICT
# =========================
predictions = model.predict(scaled_inputs)
labels = le.inverse_transform(predictions)

# Probabilities (confidence)
probs = model.predict_proba(scaled_inputs)

# =========================
# DISPLAY RESULTS
# =========================
for i in range(len(test_inputs)):
    print(f"\n🔹 Input {i+1}")
    print("Prediction:", labels[i])
    
    print("Confidence:")
    for j, class_label in enumerate(le.classes_):
        print(f"  {class_label}: {probs[i][j]:.3f}")


🔹 Input 1
Prediction: Healthy
Confidence:
  Healthy: 0.909
  Light Damage: 0.069
  Moderate Damage: 0.017
  Severe Damage: 0.005

🔹 Input 2
Prediction: Light Damage
Confidence:
  Healthy: 0.391
  Light Damage: 0.524
  Moderate Damage: 0.035
  Severe Damage: 0.051

🔹 Input 3
Prediction: Light Damage
Confidence:
  Healthy: 0.306
  Light Damage: 0.372
  Moderate Damage: 0.272
  Severe Damage: 0.050
